# py-nusantara Data Science Demo

This notebook demonstrates how to use the `py-nusantara` library directly in a Python data science workflow (such as inside Jupyter, Google Colab, or VS Code) without requiring any database connections.

### 1. Setup & Imports

In [ ]:
import py_nusantara as nus
import pandas as pd

### 2. Direct Data Access (No Database Required)

We can query and browse the administrative regions directly from memory. The package streams the gzipped CSV files in the background.

In [ ]:
# Fetch all provinces
provinces = nus.provinces()
print(f"Total Provinces: {len(provinces)}")

# Find a specific province by ID (Aceh = '11')
aceh = nus.find_province("11")
print(f"Province Found: {aceh.name} (Capital: {aceh.capital})")
print(f"Population: {aceh.population:,} | Area: {aceh.area:,} km²")

### 3. Traversal (Province -> Regencies -> Districts -> Villages)

Relationships are loaded dynamically and traverse downstream levels smoothly. Village loading is **lazy-loaded & partitioned** by Province ID to minimize memory overhead.

In [ ]:
# Traverse regencies
regencies = aceh.regencies
print(f"Regencies in {aceh.name}: {len(regencies)}")

first_regency = regencies[0]
print(f"First Regency: {first_regency.name} (ID: {first_regency.id})")

# Traverse districts
districts = first_regency.districts
print(f"Districts in {first_regency.name}: {len(districts)}")

first_district = districts[0]
print(f"First District: {first_district.name} (ID: {first_district.id})")

# Traverse villages (partitioned load)
villages = first_district.villages
print(f"Villages in {first_district.name}: {len(villages)}")
if villages:
    print(f"First Village: {villages[0].name} (Postal Code: {villages[0].postal_code})")

### 4. Substring Search

Search administrative names case-insensitively across all levels. Scanning is optimized and terminates once limits are met.

In [ ]:
results = nus.search("Bakongan")
for level, records in results.items():
    if records:
        print(f"\nMatches in [{level.upper()}]:")
        for r in records:
            print(f"  - ID: {r.id} | Name: {r.name}")

### 5. On-Demand Geographic Boundaries (GIS)

Download shapefiles on-demand from the CDN and verify their integrity dynamically using SHA-256 checksums.

In [ ]:
# Enable the boundary column in configuration dynamically
nus.init({
    "columns": {
        "provinces": {
            "boundary": {"name": "boundary", "enabled": True}
        }
    }
})

print("Downloading provinces boundary...")
nus.download_boundaries(levels="provinces")

In [ ]:
# Re-fetch the province and check boundary
aceh_boundary = nus.find_province("11")
print("Raw JSON coordinates (truncated):")
print(aceh_boundary.boundary[:150] + "...")

# Format to Well-Known Text (WKT) standard for spatial databases
wkt = nus.json_to_wkt(aceh_boundary.boundary)
print("\nWKT Format (truncated):")
print(wkt[:150] + "...")

### 6. Pandas DataFrames Export

Convert results to Pandas DataFrames for quick data processing and visualization.

In [ ]:
# Export all provinces as DataFrame
df_provinces = nus.provinces_df()
df_provinces.head()

In [ ]:
# Export regencies in Province 11 (Aceh) as DataFrame
df_regencies = nus.regencies_df("11")
df_regencies.head()